# IVF-TQ — ScaNN Baseline (Colab)

This notebook runs Google's ScaNN (anisotropic learned quantization, Guo et al. 2020) as an additional baseline for the IVF-TQ NeurIPS 2026 submission. ScaNN's `pip` package is **Linux-only**, which is why it lives outside the main `reproduce_paper.py` pipeline.

**Outputs:** `scann_results.json` — copy this back into `experiments/` of the local repo.

**Runtime:** ~20–30 min on a Colab CPU runtime (T4 GPU not used).

## 1. Install dependencies

**Expected noisy output:** pip will print dependency-conflict warnings about `tf-keras`, `ydf-tf`, and `tensorflow-text` because ScaNN pulls in tensorflow 2.18.1 while Colab's pre-installed packages expect 2.20.x. **These warnings are harmless** — ScaNN bundles its own TF and does not use those packages. As long as `import scann` succeeds in the next cell, you are fine.

In [ ]:
!pip install -q scann==1.3.5 faiss-cpu==1.13.0 numpy h5py

## 2. Sanity check ScaNN imports

ScaNN does not expose a `__version__` attribute (this was the AttributeError in the previous notebook). We instead verify the import succeeded by running a tiny smoke test.

In [ ]:
import scann, numpy as np, faiss, time, json
print('scann module path:', scann.__file__)
print('faiss:', faiss.__version__)
print('numpy:', np.__version__)

# Smoke test: build a tiny ScaNN index and run one query
_db = np.random.randn(10000, 64).astype(np.float32)
_db = _db / np.linalg.norm(_db, axis=1, keepdims=True)
_searcher = (
    scann.scann_ops_pybind.builder(_db, 10, 'dot_product')
    .tree(num_leaves=100, num_leaves_to_search=10, training_sample_size=10000)
    .score_ah(2, anisotropic_quantization_threshold=0.2)
    .reorder(50)
    .build()
)
_q = _db[:5]
_I, _D = _searcher.search_batched(_q, leaves_to_search=10, final_num_neighbors=10)
assert _I.shape == (5, 10), f'unexpected shape {_I.shape}'
print('ScaNN smoke test OK — build, search, top-10 returned.')

## 3. Helpers (mirror experiments/scann_baseline.py)

In [ ]:
def normalize(v):
    v = np.ascontiguousarray(v.astype(np.float32))
    norms = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.maximum(norms, 1e-8)

def compute_recall(gt, pred, k=10):
    hits = 0
    for i in range(gt.shape[0]):
        hits += len(set(gt[i, :k]) & set(pred[i, :k]))
    return hits / (gt.shape[0] * k)

def gt_faiss(v, q, k=10):
    v_, q_ = normalize(v), normalize(q)
    flat = faiss.IndexFlatIP(v_.shape[1]); flat.add(v_)
    _, I = flat.search(q_, k); return I

def build_scann(db, num_leaves=2000, anisotropic_threshold=0.2,
                training_sample_size=250_000, reorder_n=100):
    db = normalize(db); n = db.shape[0]
    return (scann.scann_ops_pybind.builder(db, 10, 'dot_product')
        .tree(num_leaves=num_leaves,
              num_leaves_to_search=num_leaves // 20,
              training_sample_size=min(training_sample_size, n))
        .score_ah(dimensions_per_block=2,
                  anisotropic_quantization_threshold=anisotropic_threshold)
        .reorder(reorder_n)
        .build())

def run_scann_sweep(db, queries, gt, num_leaves=2000,
                    leaves_to_search=(20, 50, 100, 200, 400)):
    t0 = time.time(); searcher = build_scann(db, num_leaves=num_leaves)
    print(f'  build={time.time()-t0:.1f}s')
    queries_n = normalize(queries); nq = queries_n.shape[0]
    out = {}
    for L in leaves_to_search:
        searcher.search_batched(queries_n[:10], leaves_to_search=L)
        ts = []
        I = None
        for _ in range(3):
            t0 = time.time()
            I, _ = searcher.search_batched(queries_n, leaves_to_search=L, final_num_neighbors=10)
            ts.append(time.time() - t0)
        t = float(np.median(ts))
        r = compute_recall(gt, I, 10)
        n, dim = db.shape
        codes = n * (dim // 2)
        codebook = 256 * (dim // 2) * 2 * 4
        coarse = num_leaves * dim * 4
        reorder = n * dim * 4
        comp_mb = (codes + codebook + coarse) / (1024 * 1024)
        total_mb = comp_mb + reorder / (1024 * 1024)
        out[f'leaves{L}'] = {'recall10': round(r*100,1),
                              'qps': round(nq/t),
                              'latency_ms': round(t*1000,1),
                              'memory_mb': round(comp_mb,1),
                              'total_memory_mb': round(total_mb,1),
                              'compression': f'{(n*dim*4)/(codes+codebook+coarse):.1f}x',
                              'training': 'Anisotropic AH codebook + tree partition'}
        print(f'    L={L}: R@10={r:.1%}  {nq/t:.0f} QPS')
    return out

## 4. Load datasets

Uses `ann-benchmarks` HDF5 mirrors so we don't need the full repo on Colab.

In [ ]:
import urllib.request, h5py, os, shutil

URLS = {
    'sift1m': 'http://ann-benchmarks.com/sift-128-euclidean.hdf5',
    'deep1m': 'http://ann-benchmarks.com/deep-image-96-angular.hdf5',
}

def load_dataset(name):
    path = f'/content/{name}.hdf5'
    if not os.path.exists(path):
        print(f'Downloading {name} ...')
        # ann-benchmarks.com 403s urllib's default User-Agent; spoof Mozilla
        req = urllib.request.Request(
            URLS[name],
            headers={'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64)'}
        )
        with urllib.request.urlopen(req, timeout=120) as response, open(path, 'wb') as f:
            shutil.copyfileobj(response, f)
        print(f'  saved {path} ({os.path.getsize(path)/1e6:.0f} MB)')
    with h5py.File(path, 'r') as f:
        train = np.array(f['train'], dtype=np.float32)
        test = np.array(f['test'], dtype=np.float32)
    nq = min(test.shape[0], 10_000)
    return train[:1_000_000], test[:nq]

## 5. Run ScaNN sweep on SIFT-1M and Deep-1M

In [ ]:
results = {}
for ds in ['sift1m', 'deep1m']:
    print(f'\n=== {ds.upper()} ===')
    v, q = load_dataset(ds)
    print(f'  n={v.shape[0]}  dim={v.shape[1]}  nq={q.shape[0]}')
    print('  computing GT...')
    gt = gt_faiss(v, q, k=10)
    sub = run_scann_sweep(v, q, gt,
                          num_leaves=2000,
                          leaves_to_search=(20, 50, 100, 200, 400))
    results[f'scann_{ds}'] = sub

with open('/content/scann_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved /content/scann_results.json')
print(json.dumps(results, indent=2))

## 6. Download results back to local repo

After this cell, copy the file into your local `experiments/` directory.

In [ ]:
from google.colab import files
files.download('/content/scann_results.json')